In [5]:
import os
import soccerdata as sd
import pandas as pd
import pyspark.sql.functions as f
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17"

spark = configure_spark_with_delta_pip(
    SparkSession.builder.master("local[*]")
    .appName("exploration")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
).getOrCreate()

26/07/18 10:44:52 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$driverEndpoint(BlockManagerMasterEndpoint.scala:131)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.isExecutorAlive$lzycompute$1(BlockManagerMasterEndpoint.scala:707)
	at org.apache.spark.storage.BlockManagerMasterE

In [ ]:
fbref = sd.FBref(leagues=["ENG-Premier League"], seasons=["2022"])

[06/07/26 13:10:34] INFO     Saving cached data to /Users/shreyastelkar/soccerdata/data/FBref        ]8;id=8814547;file:///Users/shreyastelkar/Documents/Git/football_analysis/football_analysis/.venv/lib/python3.13/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=8814548;file:///Users/shreyastelkar/Documents/Git/football_analysis/football_analysis/.venv/lib/python3.13/site-packages/soccerdata/_common.py#250\250]8;;\

In [ ]:
df_team_season_stats = fbref.read_team_season_stats(stat_type="standard")
df_team_season_stats.head(5)

players_used   Age  Poss Playing Time                  Performance                                  Per 90 Minutes                           \
                                                                         MP Starts   Min 90s         Gls Ast  G+A G-PK PK PKatt CrdY CrdR            Gls   Ast   G+A  G-PK G+A-PK   
league             season team                                                                                                                                                      
ENG-Premier League 2223   Arsenal               26  24.7  59.3           38    418  3420  38          84  64  148   81  3     4   51    0           2.21  1.68  3.89  2.13   3.82   
                          Aston Villa           26  27.0  49.3           38    418  3420  38          49  35   84   46  3     4   80    1           1.29  0.92  2.21  1.21   2.13   
                          Bournemouth           31  26.3  40.4           38    418  3420  38          37  24   61   37  0     0   68    0           0.97  0.63  1.61  0.97   1.61   
                          Brentford             25  26.2  43.8           38    418  3420  38          56  36   92   49  7     8   57    1           1.47  0.95  2.42  1.29   2.24   
                          Brighton              29  26.3  60.2           38    418  3420  38          68  46  114   62  6     6   58    0           1.79  1.21   3.0  1.63   2.84   

                                                                                     url  
                                                                                          
league             season team                                                            
ENG-Premier League 2223   Arsenal            /en/squads/18bb7c10/2022-2023/Arsenal-Stats  
                          Aston Villa    /en/squads/8602292d/2022-2023/Aston-Villa-Stats  
                          Bournemouth    /en/squads/4ba7cbea/2022-2023/Bournemouth-Stats  
                          Brentford        /en/squads/cd051869/2022-2023/Brentford-Stats  
                          Brighton     /en/squads/d07537b9/2022-2023/Brighton-and-Hov...

In [ ]:
df_fbref_schedule = spark.read.format("delta").load("../data/bronze/fbref_schedule")
df_fbref_schedule.show(5)

26/06/21 12:33:32 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------------+------+--------------------+----+---+-------------------+-------------+----------+-----+-------------------+----------+--------------------+----------------+--------------------+-----+----------+--------+--------------------+
|        league|season|                game|week|day|               date|         time| home_team|score|          away_team|attendance|               venue|         referee|        match_report|notes|     round| game_id|         ingested_at|
+--------------+------+--------------------+----+---+-------------------+-------------+----------+-----+-------------------+----------+--------------------+----------------+--------------------+-----+----------+--------+--------------------+
|GER-Bundesliga|  2425|2024-08-23 Gladba...| 1.0|Fri|2024-08-23 00:00:00|20:30 (00:00)|  Gladbach|  2–3|         Leverkusen|   54042.0|Stadion im Boruss...| Robert Schröder|/en/matches/d42e5...|  nan|Bundesliga|d42e53df|2026-06-12 17:08:...|
|GER-Bundesliga|  2425|2024-08-2

In [4]:
df_fbref_player_season_stats_standard = spark.read.format("delta").load("../data/bronze/fbref_player_season_stats_standard")
df_fbref_player_season_stats_standard.show(5)

26/06/21 17:43:30 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-----------+------+-------+----------------+------+---+----+------+---------------+-------------------+----------------+----------------+---------------+---------------+---------------+----------------+--------------+-----------------+----------------+----------------+------------------+------------------+------------------+-------------------+---------------------+--------------------+
|     league|season|   team|          player|nation|pos| age|  born|playing_time_mp|playing_time_starts|playing_time_min|playing_time_90s|performance_gls|performance_ast|performance_g_a|performance_g_pk|performance_pk|performance_pkatt|performance_crdy|performance_crdr|per_90_minutes_gls|per_90_minutes_ast|per_90_minutes_g_a|per_90_minutes_g_pk|per_90_minutes_g_a_pk|         ingested_at|
+-----------+------+-------+----------------+------+---+----+------+---------------+-------------------+----------------+----------------+---------------+---------------+---------------+----------------+--------------+

In [3]:
df_fbref_player_season_stats_standard = spark.read.format("delta").load("../data/bronze/fbref_player_season_stats_standard")
# df_fbref_player_season_stats_standard.show(5)
df_fbref_player_season_stats_standard.printSchema()

root
 |-- league: string (nullable = true)
 |-- season: string (nullable = true)
 |-- team: string (nullable = true)
 |-- player: string (nullable = true)
 |-- nation: string (nullable = true)
 |-- pos: string (nullable = true)
 |-- age: double (nullable = true)
 |-- born: double (nullable = true)
 |-- playing_time_mp: long (nullable = true)
 |-- playing_time_starts: long (nullable = true)
 |-- playing_time_min: long (nullable = true)
 |-- playing_time_90s: double (nullable = true)
 |-- performance_gls: long (nullable = true)
 |-- performance_ast: long (nullable = true)
 |-- performance_g_a: long (nullable = true)
 |-- performance_g_pk: long (nullable = true)
 |-- performance_pk: long (nullable = true)
 |-- performance_pkatt: long (nullable = true)
 |-- performance_crdy: long (nullable = true)
 |-- performance_crdr: long (nullable = true)
 |-- per_90_minutes_gls: double (nullable = true)
 |-- per_90_minutes_ast: double (nullable = true)
 |-- per_90_minutes_g_a: double (nullable = true)

In [5]:
df_fbref_player_season_stats_shooting = spark.read.format("delta").load("../data/bronze/fbref_player_season_stats_shooting")
df_fbref_player_season_stats_shooting.show(5)

+-----------+------+------+-----------------+------+-----+----+------+-----+------------+-----------+------------+--------------+--------------+---------------+-------------+--------------+-----------+--------------+--------------------+
|     league|season|  team|           player|nation|  pos| age|  born|n_90s|standard_gls|standard_sh|standard_sot|standard_sot_1|standard_sh_90|standard_sot_90|standard_g_sh|standard_g_sot|standard_pk|standard_pkatt|         ingested_at|
+-----------+------+------+-----------------+------+-----+----+------+-----+------------+-----------+------------+--------------+--------------+---------------+-------------+--------------+-----------+--------------+--------------------+
|FRA-Ligue 1|  2425|Monaco|     Eliot Matazo|   BEL|   MF|22.0|2002.0|  0.8|           0|          0|           0|           NaN|           0.0|            0.0|          NaN|           NaN|          0|             0|2026-06-12 18:23:...|
|FRA-Ligue 1|  2425|Monaco|  Folarin Balogun|   

In [6]:
df_fbref_player_season_stats_shooting.printSchema()

root
 |-- league: string (nullable = true)
 |-- season: string (nullable = true)
 |-- team: string (nullable = true)
 |-- player: string (nullable = true)
 |-- nation: string (nullable = true)
 |-- pos: string (nullable = true)
 |-- age: double (nullable = true)
 |-- born: double (nullable = true)
 |-- n_90s: double (nullable = true)
 |-- standard_gls: long (nullable = true)
 |-- standard_sh: long (nullable = true)
 |-- standard_sot: long (nullable = true)
 |-- standard_sot_1: double (nullable = true)
 |-- standard_sh_90: double (nullable = true)
 |-- standard_sot_90: double (nullable = true)
 |-- standard_g_sh: double (nullable = true)
 |-- standard_g_sot: double (nullable = true)
 |-- standard_pk: long (nullable = true)
 |-- standard_pkatt: long (nullable = true)
 |-- ingested_at: timestamp (nullable = true)



In [7]:
df_fbref_player_season_stats_playing_time = spark.read.format("delta").load("../data/bronze/fbref_player_season_stats_playing_time")
df_fbref_player_season_stats_playing_time.show(5)

+------------------+------+------------+----------------+------+---+----+------+---------------+----------------+------------------+------------------+----------------+-------------+---------------+------------+---------+-----------+----------+----------------+----------------+-----------------+------------+---------------+-------------------+--------------------+
|            league|season|        team|          player|nation|pos| age|  born|playing_time_mp|playing_time_min|playing_time_mn_mp|playing_time_min_1|playing_time_90s|starts_starts|starts_mn_start|starts_compl|subs_subs|subs_mn_sub|subs_unsub|team_success_ppm|team_success_ong|team_success_onga|team_success|team_success_90|team_success_on_off|         ingested_at|
+------------------+------+------------+----------------+------+---+----+------+---------------+----------------+------------------+------------------+----------------+-------------+---------------+------------+---------+-----------+----------+----------------+-----

In [8]:
df_fbref_player_season_stats_playing_time.printSchema()

root
 |-- league: string (nullable = true)
 |-- season: string (nullable = true)
 |-- team: string (nullable = true)
 |-- player: string (nullable = true)
 |-- nation: string (nullable = true)
 |-- pos: string (nullable = true)
 |-- age: double (nullable = true)
 |-- born: double (nullable = true)
 |-- playing_time_mp: long (nullable = true)
 |-- playing_time_min: double (nullable = true)
 |-- playing_time_mn_mp: double (nullable = true)
 |-- playing_time_min_1: double (nullable = true)
 |-- playing_time_90s: double (nullable = true)
 |-- starts_starts: long (nullable = true)
 |-- starts_mn_start: double (nullable = true)
 |-- starts_compl: long (nullable = true)
 |-- subs_subs: long (nullable = true)
 |-- subs_mn_sub: double (nullable = true)
 |-- subs_unsub: long (nullable = true)
 |-- team_success_ppm: double (nullable = true)
 |-- team_success_ong: double (nullable = true)
 |-- team_success_onga: double (nullable = true)
 |-- team_success: double (nullable = true)
 |-- team_success

In [ ]:
df_fbref_player_season_stats_misc = spark.read.format("delta").load("../data/bronze/fbref_player_season_stats_misc")
df_fbref_player_season_stats_misc.show(5)

26/06/23 18:31:08 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+------------------+------+-------+-------------+------+-----+----+------+-----+----------------+----------------+-----------------+---------------+---------------+---------------+---------------+---------------+----------------+-----------------+-----------------+--------------+--------------------+
|            league|season|   team|       player|nation|  pos| age|  born|n_90s|performance_crdy|performance_crdr|performance_2crdy|performance_fls|performance_fld|performance_off|performance_crs|performance_int|performance_tklw|performance_pkwon|performance_pkcon|performance_og|         ingested_at|
+------------------+------+-------+-------------+------+-----+----+------+-----+----------------+----------------+-----------------+---------------+---------------+---------------+---------------+---------------+----------------+-----------------+-----------------+--------------+--------------------+
|ENG-Premier League|  2425|Arsenal|    Ben White|   ENG|   DF|26.0|1997.0| 13.3|              

In [ ]:
df_fbref_player_season_stats_misc.printSchema()

root
 |-- league: string (nullable = true)
 |-- season: string (nullable = true)
 |-- team: string (nullable = true)
 |-- player: string (nullable = true)
 |-- nation: string (nullable = true)
 |-- pos: string (nullable = true)
 |-- age: double (nullable = true)
 |-- born: double (nullable = true)
 |-- n_90s: double (nullable = true)
 |-- performance_crdy: long (nullable = true)
 |-- performance_crdr: long (nullable = true)
 |-- performance_2crdy: long (nullable = true)
 |-- performance_fls: long (nullable = true)
 |-- performance_fld: long (nullable = true)
 |-- performance_off: long (nullable = true)
 |-- performance_crs: long (nullable = true)
 |-- performance_int: long (nullable = true)
 |-- performance_tklw: long (nullable = true)
 |-- performance_pkwon: string (nullable = true)
 |-- performance_pkcon: string (nullable = true)
 |-- performance_og: long (nullable = true)
 |-- ingested_at: timestamp (nullable = true)



In [4]:
df_fbref_player_season_stats_keeper = spark.read.format("delta").load("../data/bronze/fbref_player_season_stats_keeper")
df_fbref_player_season_stats_keeper.show(5)

+------------------+------+-----------+-----------------+------+---+---+----+---------------+-------------------+----------------+----------------+--------------+----------------+----------------+-----------------+----------------+-------------+-------------+-------------+--------------+----------------+-------------------+-----------------+------------------+-----------------+------------------+--------------------+
|            league|season|       team|           player|nation|pos|age|born|playing_time_mp|playing_time_starts|playing_time_min|playing_time_90s|performance_ga|performance_ga90|performance_sota|performance_saves|performance_save|performance_w|performance_d|performance_l|performance_cs|performance_cs_1|penalty_kicks_pkatt|penalty_kicks_pka|penalty_kicks_pksv|penalty_kicks_pkm|penalty_kicks_save|         ingested_at|
+------------------+------+-----------+-----------------+------+---+---+----+---------------+-------------------+----------------+----------------+-----------

In [6]:
df_fbref_player_season_stats_keeper.printSchema()

root
 |-- league: string (nullable = true)
 |-- season: string (nullable = true)
 |-- team: string (nullable = true)
 |-- player: string (nullable = true)
 |-- nation: string (nullable = true)
 |-- pos: string (nullable = true)
 |-- age: long (nullable = true)
 |-- born: long (nullable = true)
 |-- playing_time_mp: long (nullable = true)
 |-- playing_time_starts: long (nullable = true)
 |-- playing_time_min: long (nullable = true)
 |-- playing_time_90s: double (nullable = true)
 |-- performance_ga: long (nullable = true)
 |-- performance_ga90: double (nullable = true)
 |-- performance_sota: long (nullable = true)
 |-- performance_saves: long (nullable = true)
 |-- performance_save: double (nullable = true)
 |-- performance_w: long (nullable = true)
 |-- performance_d: long (nullable = true)
 |-- performance_l: long (nullable = true)
 |-- performance_cs: long (nullable = true)
 |-- performance_cs_1: double (nullable = true)
 |-- penalty_kicks_pkatt: long (nullable = true)
 |-- penalty_k

In [7]:
df_fbref_seasons = spark.read.format("delta").load("../data/bronze/fbref_seasons")
df_fbref_seasons.show(5)

+------------------+------+-----------+--------------------+--------------------+
|            league|season|     format|                 url|         ingested_at|
+------------------+------+-----------+--------------------+--------------------+
|ENG-Premier League|  2425|round-robin|/en/comps/9/2024-...|2026-06-12 17:08:...|
|    GER-Bundesliga|  2425|round-robin|/en/comps/20/2024...|2026-06-12 17:08:...|
|       FRA-Ligue 1|  2425|round-robin|/en/comps/13/2024...|2026-06-12 17:08:...|
|       ESP-La Liga|  2425|round-robin|/en/comps/12/2024...|2026-06-12 17:08:...|
|       ITA-Serie A|  2425|round-robin|/en/comps/11/2024...|2026-06-12 17:08:...|
+------------------+------+-----------+--------------------+--------------------+



In [8]:
df_fbref_seasons.printSchema()

root
 |-- league: string (nullable = true)
 |-- season: string (nullable = true)
 |-- format: string (nullable = true)
 |-- url: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)



In [2]:
df_fbref_team_match_stats_keeper = spark.read.format("delta").load("../data/bronze/fbref_team_match_stats_keeper")
df_fbref_team_match_stats_keeper.show(5)

26/07/05 18:25:47 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-----------+------+------+--------------------+-------------------+-----------+---+-----+------+---+---+-------------+----------------+--------------+-----------------+----------------+--------------+-------------------+-----------------+------------------+-----------------+--------+--------------------+--------------------+
|     league|season|  team|                game|               date|      round|day|venue|result| gf| ga|     opponent|performance_sota|performance_ga|performance_saves|performance_save|performance_cs|penalty_kicks_pkatt|penalty_kicks_pka|penalty_kicks_pksv|penalty_kicks_pkm|    time|        match_report|         ingested_at|
+-----------+------+------+--------------------+-------------------+-----------+---+-----+------+---+---+-------------+----------------+--------------+-----------------+----------------+--------------+-------------------+-----------------+------------------+-----------------+--------+--------------------+--------------------+
|ESP-La Liga|  2

In [3]:
df_fbref_team_match_stats_keeper.printSchema()

root
 |-- league: string (nullable = true)
 |-- season: string (nullable = true)
 |-- team: string (nullable = true)
 |-- game: string (nullable = true)
 |-- date: timestamp (nullable = true)
 |-- round: string (nullable = true)
 |-- day: string (nullable = true)
 |-- venue: string (nullable = true)
 |-- result: string (nullable = true)
 |-- gf: string (nullable = true)
 |-- ga: string (nullable = true)
 |-- opponent: string (nullable = true)
 |-- performance_sota: double (nullable = true)
 |-- performance_ga: double (nullable = true)
 |-- performance_saves: double (nullable = true)
 |-- performance_save: double (nullable = true)
 |-- performance_cs: double (nullable = true)
 |-- penalty_kicks_pkatt: double (nullable = true)
 |-- penalty_kicks_pka: double (nullable = true)
 |-- penalty_kicks_pksv: double (nullable = true)
 |-- penalty_kicks_pkm: double (nullable = true)
 |-- time: string (nullable = true)
 |-- match_report: string (nullable = true)
 |-- ingested_at: timestamp (nullable

In [5]:
df_fbref_team_match_stats_keeper.filter((f.col("team") == "Celta Vigo") & (f.col("date") == "2024-08-16 00:00:00")).show(5)

+-----------+------+----------+--------------------+-------------------+-----------+---+-----+------+---+---+--------+----------------+--------------+-----------------+----------------+--------------+-------------------+-----------------+------------------+-----------------+--------+--------------------+--------------------+
|     league|season|      team|                game|               date|      round|day|venue|result| gf| ga|opponent|performance_sota|performance_ga|performance_saves|performance_save|performance_cs|penalty_kicks_pkatt|penalty_kicks_pka|penalty_kicks_pksv|penalty_kicks_pkm|    time|        match_report|         ingested_at|
+-----------+------+----------+--------------------+-------------------+-----------+---+-----+------+---+---+--------+----------------+--------------+-----------------+----------------+--------------+-------------------+-----------------+------------------+-----------------+--------+--------------------+--------------------+
|ESP-La Liga|  2425

In [2]:
df_fbref_team_match_stats_misc = spark.read.format("delta").load("../data/bronze/fbref_team_match_stats_misc")
df_fbref_team_match_stats_misc.show(5)

26/07/12 16:45:37 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-----------+------+---------------+--------------------+-------------------+------------+---+-----+------+---+---+-------------+----------------+----------------+-----------------+---------------+---------------+---------------+---------------+---------------+----------------+-----------------+-----------------+--------------+--------+--------------------+--------------------+
|     league|season|           team|                game|               date|       round|day|venue|result| gf| ga|     opponent|performance_crdy|performance_crdr|performance_2crdy|performance_fls|performance_fld|performance_off|performance_crs|performance_int|performance_tklw|performance_pkwon|performance_pkcon|performance_og|    time|        match_report|         ingested_at|
+-----------+------+---------------+--------------------+-------------------+------------+---+-----+------+---+---+-------------+----------------+----------------+-----------------+---------------+---------------+---------------+---------

In [3]:
df_fbref_team_match_stats_misc.printSchema()

root
 |-- league: string (nullable = true)
 |-- season: string (nullable = true)
 |-- team: string (nullable = true)
 |-- game: string (nullable = true)
 |-- date: timestamp (nullable = true)
 |-- round: string (nullable = true)
 |-- day: string (nullable = true)
 |-- venue: string (nullable = true)
 |-- result: string (nullable = true)
 |-- gf: string (nullable = true)
 |-- ga: string (nullable = true)
 |-- opponent: string (nullable = true)
 |-- performance_crdy: long (nullable = true)
 |-- performance_crdr: long (nullable = true)
 |-- performance_2crdy: long (nullable = true)
 |-- performance_fls: double (nullable = true)
 |-- performance_fld: double (nullable = true)
 |-- performance_off: double (nullable = true)
 |-- performance_crs: double (nullable = true)
 |-- performance_int: double (nullable = true)
 |-- performance_tklw: double (nullable = true)
 |-- performance_pkwon: string (nullable = true)
 |-- performance_pkcon: string (nullable = true)
 |-- performance_og: double (null

In [4]:
df_fbref_team_match_stats_schedule = spark.read.format("delta").load("../data/bronze/fbref_team_match_stats_schedule")
df_fbref_team_match_stats_schedule.show(5)

+------------------+------+-------+--------------------+-------------------+--------+-----------+---+-----+------+---+---+-----------------+----+----------+---------------+---------+-------------+--------------+--------------------+-----+--------------------+
|            league|season|   team|                game|               date|    time|      round|day|venue|result| gf| ga|         opponent|poss|attendance|        captain|formation|opp_formation|       referee|        match_report|notes|         ingested_at|
+------------------+------+-------+--------------------+-------------------+--------+-----------+---+-----+------+---+---+-----------------+----+----------+---------------+---------+-------------+--------------+--------------------+-----+--------------------+
|ENG-Premier League|  2425|Arsenal|2024-08-17 Arsena...|2024-08-17 00:00:00|15:00:00|Matchweek 1|Sat| Home|     W|  2|  0|           Wolves|53.0|   60261.0|Martin Ødegaard|    4-3-3|      4-2-3-1|Jarred Gillett|/en/match

In [5]:
df_fbref_team_match_stats_schedule.printSchema()

root
 |-- league: string (nullable = true)
 |-- season: string (nullable = true)
 |-- team: string (nullable = true)
 |-- game: string (nullable = true)
 |-- date: timestamp (nullable = true)
 |-- time: string (nullable = true)
 |-- round: string (nullable = true)
 |-- day: string (nullable = true)
 |-- venue: string (nullable = true)
 |-- result: string (nullable = true)
 |-- gf: string (nullable = true)
 |-- ga: string (nullable = true)
 |-- opponent: string (nullable = true)
 |-- poss: double (nullable = true)
 |-- attendance: double (nullable = true)
 |-- captain: string (nullable = true)
 |-- formation: string (nullable = true)
 |-- opp_formation: string (nullable = true)
 |-- referee: string (nullable = true)
 |-- match_report: string (nullable = true)
 |-- notes: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)



In [6]:
df_fbref_team_match_stats_shooting = spark.read.format("delta").load("../data/bronze/fbref_team_match_stats_shooting")
df_fbref_team_match_stats_shooting.show(5)

+-----------+------+-----+--------------------+-------------------+------------+---+-----+------+---+---+--------+------------+-----------+------------+--------------+-------------+--------------+-----------+--------------+--------+--------------------+--------------------+
|     league|season| team|                game|               date|       round|day|venue|result| gf| ga|opponent|standard_gls|standard_sh|standard_sot|standard_sot_1|standard_g_sh|standard_g_sot|standard_pk|standard_pkatt|    time|        match_report|         ingested_at|
+-----------+------+-----+--------------------+-------------------+------------+---+-----+------+---+---+--------+------------+-----------+------------+--------------+-------------+--------------+-----------+--------------+--------+--------------------+--------------------+
|ITA-Serie A|  2425|Monza|2024-10-30 Atalan...|2024-10-30 00:00:00|Matchweek 10|Wed| Away|     L|  0|  2|Atalanta|           0|        4.0|         3.0|          75.0|        

In [7]:
df_fbref_team_match_stats_shooting.printSchema()

root
 |-- league: string (nullable = true)
 |-- season: string (nullable = true)
 |-- team: string (nullable = true)
 |-- game: string (nullable = true)
 |-- date: timestamp (nullable = true)
 |-- round: string (nullable = true)
 |-- day: string (nullable = true)
 |-- venue: string (nullable = true)
 |-- result: string (nullable = true)
 |-- gf: string (nullable = true)
 |-- ga: string (nullable = true)
 |-- opponent: string (nullable = true)
 |-- standard_gls: long (nullable = true)
 |-- standard_sh: double (nullable = true)
 |-- standard_sot: double (nullable = true)
 |-- standard_sot_1: double (nullable = true)
 |-- standard_g_sh: double (nullable = true)
 |-- standard_g_sot: double (nullable = true)
 |-- standard_pk: long (nullable = true)
 |-- standard_pkatt: long (nullable = true)
 |-- time: string (nullable = true)
 |-- match_report: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)



In [8]:
df_fbref_team_season_stats_keeper = spark.read.format("delta").load("../data/bronze/fbref_team_season_stats_keeper")
df_fbref_team_season_stats_keeper.show(5)

+--------------+------+-------------+------------+---------------+-------------------+----------------+----------------+--------------+----------------+----------------+-----------------+----------------+-------------+-------------+-------------+--------------+----------------+-------------------+-----------------+------------------+-----------------+------------------+--------------------+--------------------+
|        league|season|         team|players_used|playing_time_mp|playing_time_starts|playing_time_min|playing_time_90s|performance_ga|performance_ga90|performance_sota|performance_saves|performance_save|performance_w|performance_d|performance_l|performance_cs|performance_cs_1|penalty_kicks_pkatt|penalty_kicks_pka|penalty_kicks_pksv|penalty_kicks_pkm|penalty_kicks_save|                 url|         ingested_at|
+--------------+------+-------------+------------+---------------+-------------------+----------------+----------------+--------------+----------------+----------------+-

In [10]:
df_fbref_team_season_stats_keeper.printSchema()

root
 |-- league: string (nullable = true)
 |-- season: string (nullable = true)
 |-- team: string (nullable = true)
 |-- players_used: long (nullable = true)
 |-- playing_time_mp: long (nullable = true)
 |-- playing_time_starts: long (nullable = true)
 |-- playing_time_min: long (nullable = true)
 |-- playing_time_90s: long (nullable = true)
 |-- performance_ga: long (nullable = true)
 |-- performance_ga90: double (nullable = true)
 |-- performance_sota: long (nullable = true)
 |-- performance_saves: long (nullable = true)
 |-- performance_save: double (nullable = true)
 |-- performance_w: long (nullable = true)
 |-- performance_d: long (nullable = true)
 |-- performance_l: long (nullable = true)
 |-- performance_cs: long (nullable = true)
 |-- performance_cs_1: double (nullable = true)
 |-- penalty_kicks_pkatt: long (nullable = true)
 |-- penalty_kicks_pka: long (nullable = true)
 |-- penalty_kicks_pksv: long (nullable = true)
 |-- penalty_kicks_pkm: long (nullable = true)
 |-- penal

In [3]:
df_fbref_team_season_stats_misc = spark.read.format("delta").load("../data/bronze/fbref_team_season_stats_misc")
df_fbref_team_season_stats_misc.show(5)

26/07/18 10:07:52 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+------------------+------+-----------+------------+-----+----------------+----------------+-----------------+---------------+---------------+---------------+---------------+---------------+----------------+-----------------+-----------------+--------------+--------------------+--------------------+
|            league|season|       team|players_used|n_90s|performance_crdy|performance_crdr|performance_2crdy|performance_fls|performance_fld|performance_off|performance_crs|performance_int|performance_tklw|performance_pkwon|performance_pkcon|performance_og|                 url|         ingested_at|
+------------------+------+-----------+------------+-----+----------------+----------------+-----------------+---------------+---------------+---------------+---------------+---------------+----------------+-----------------+-----------------+--------------+--------------------+--------------------+
|ENG-Premier League|  2425|    Arsenal|          25|   38|              70|               6|     

In [ ]:
df_fbref_team_season_stats_misc.printSchema()

root
 |-- league: string (nullable = true)
 |-- season: string (nullable = true)
 |-- team: string (nullable = true)
 |-- players_used: long (nullable = true)
 |-- n_90s: long (nullable = true)
 |-- performance_crdy: long (nullable = true)
 |-- performance_crdr: long (nullable = true)
 |-- performance_2crdy: long (nullable = true)
 |-- performance_fls: long (nullable = true)
 |-- performance_fld: long (nullable = true)
 |-- performance_off: long (nullable = true)
 |-- performance_crs: long (nullable = true)
 |-- performance_int: long (nullable = true)
 |-- performance_tklw: long (nullable = true)
 |-- performance_pkwon: long (nullable = true)
 |-- performance_pkcon: long (nullable = true)
 |-- performance_og: long (nullable = true)
 |-- url: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)



26/07/18 10:52:02 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:81)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:674)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1363)
	at org.apache.spark.executor.Executor.$anonfun$heartbeater$1(Executor.scala:356)
	at scala.runtime.java8.JFunction0$mcV$sp.apply(JFunction0$mcV$sp.scala:18)
	at org.apache.spark.util.Utils$.logUncaughtExceptions(Utils.scala:1941

In [9]:
df_fbref_team_season_stats_playing_time = spark.read.format("delta").load("../data/bronze/fbref_team_season_stats_playing_time")
df_fbref_team_season_stats_playing_time.show(5)

+-----------+------+-------------+------------+----+---------------+----------------+------------------+------------------+----------------+-------------+---------------+------------+---------+-----------+----------+----------------+----------------+-----------------+------------+---------------+--------------------+--------------------+
|     league|season|         team|players_used| age|playing_time_mp|playing_time_min|playing_time_mn_mp|playing_time_min_1|playing_time_90s|starts_starts|starts_mn_start|starts_compl|subs_subs|subs_mn_sub|subs_unsub|team_success_ppm|team_success_ong|team_success_onga|team_success|team_success_90|                 url|         ingested_at|
+-----------+------+-------------+------------+----+---------------+----------------+------------------+------------------+----------------+-------------+---------------+------------+---------+-----------+----------+----------------+----------------+-----------------+------------+---------------+--------------------+--

In [ ]:
df_fbref_team_season_stats_playing_time.printSchema()

root
 |-- league: string (nullable = true)
 |-- season: string (nullable = true)
 |-- team: string (nullable = true)
 |-- players_used: long (nullable = true)
 |-- age: double (nullable = true)
 |-- playing_time_mp: long (nullable = true)
 |-- playing_time_min: long (nullable = true)
 |-- playing_time_mn_mp: long (nullable = true)
 |-- playing_time_min_1: long (nullable = true)
 |-- playing_time_90s: long (nullable = true)
 |-- starts_starts: long (nullable = true)
 |-- starts_mn_start: long (nullable = true)
 |-- starts_compl: long (nullable = true)
 |-- subs_subs: long (nullable = true)
 |-- subs_mn_sub: long (nullable = true)
 |-- subs_unsub: long (nullable = true)
 |-- team_success_ppm: double (nullable = true)
 |-- team_success_ong: long (nullable = true)
 |-- team_success_onga: long (nullable = true)
 |-- team_success: long (nullable = true)
 |-- team_success_90: double (nullable = true)
 |-- url: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)



26/07/18 10:53:22 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$driverEndpoint(BlockManagerMasterEndpoint.scala:131)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.isExecutorAlive$lzycompute$1(BlockManagerMasterEndpoint.scala:707)
	at org.apache.spark.storage.BlockManagerMasterE

In [ ]:
df_fbref_match_stats_shoo = spark.read.format("delta").load("../data/bronze/fbref_team_season_stats_playing_time")
df_fbref_team_season_stats_playing_time.show(5)